In [0]:
%run /Users/jamiquecampbell@outlook.com/Databricks-Demos/utils/helper_utils

In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
%skip
#We found Input files are messy and have no header, had to start at row 8 
#can skip
df = spark.read.option('skipRows', 7).csv(
    '/Volumes/jc_demoworkspace/datahive/airline_departures',
    header =True,
    inferSchema =True
)
display(df.orderBy(df['Flight Number'], ascending=True).head(5))

df.printSchema() 

In [0]:
schema_new = (
StructType([
    StructField("carrier_code", StringType(), True),
    StructField("date", StringType(), True),
    StructField("flight_number", StringType(), True),
    StructField("tail_number", StringType(), True),
    StructField("destination_airport", StringType(), True),
    StructField("scheduled_departure_time", StringType(), True),
    StructField("actual_departure_time", StringType(), True),
    StructField("scheduled_elapsed_time_minutes", IntegerType(), True),
    StructField("actual_elapsed_time_minutes", IntegerType(), True),
    StructField("departure_delay_minutes", IntegerType(), True),
    StructField("wheelsoff_time", StringType(), True),
    StructField("taxiout_time_minutes", IntegerType(), True),
    StructField("delay_carrier_minutes", IntegerType(), True),
    StructField("delay_weather_minutes", IntegerType(), True),
    StructField("delay_national_aviation_system_minutes", IntegerType(), True),
    StructField("delay_security_minutes", IntegerType(), True),
    StructField("delay_late_aircraft_arrival_minutes", IntegerType(), True)
])
)



departures_df = spark.read.option('skipRows', 8).csv(
    '/Volumes/jc_demoworkspace/datahive/airline_departures',
    schema = schema_new
)
display(departures_df)

In [0]:
#need to get origin airport code from original file

flight_origin  = spark.read.option('skipRows',1).csv(
    '/Volumes/jc_demoworkspace/datahive/airline_departures',
    header =False,
    inferSchema =True
)

flight_origin = flight_origin.withColumn(
    "origin_code",
    F.regexp_extract("_c1", r"\((.*?)\)", 1)
)


origin = flight_origin.select("origin_code").first()[0]
print(origin)

In [0]:
departures_df_enriched = (
    departures_df
        .withColumn("origin_code",F.lit(origin))
        .withColumn("date", F.to_date("date","MM/dd/yyyy")) #had to change date format because of spark default date format
)
departures_df_enriched = add_meta_data_bronze(departures_df_enriched)
display(departures_df_enriched)

In [0]:
df_count = departures_df_enriched.count()
try:
    departures_df.count() == departures_df_enriched.count()
except:
    print("Dataframe count does not match, rows lost during enrichment")
else:
    print(f"Dataframe count matches {df_count} rows.")

In [0]:
table_path = get_table_config('airlines','bronze')

create_or_replace(departures_df_enriched, table_path)